In [1]:
import numpy as np, numpy.linalg as la
from coppertop.pipe import *
from bones.ts.metatypes import BType, extractConstructors
    
from coppertop.dm.core.types import darray, pylist, index, matrix, pyint
from coppertop.dm.core import to, at, shape
from coppertop.dm.core.text_report import display_table, PP, join, TR
from coppertop.dm.linalg.core import mmul, madd, col, cols, row, rows, inv, PP, TR
from coppertop.dm.pp import PP, DD, TT

# OPEN: fix import / override bug - see PP in coppertop.dm.linalg.core


In [9]:
A = [[2,3],[2,4],[3,7]] >> to >> matrix
b = [[1],[2]] >> to >> matrix
A >> typeOf >> PP
A @ b

matrix([[ 8]
 [10]
 [17]])

In [10]:
(2 * (A >> col(_,1)), 3 * (A >> col(_,2))) 

(matrix([[4]
  [4]
  [6]]),
 matrix([[ 9]
  [12]
  [21]]))

In [11]:
A  @ matrix([[2],[3]]) >> PP;

[[13] 
 [16] 
 [27]]


In [12]:
matrix([1,3], [1,2], [0,1]) @ matrix([1,0,2], [0,1,2]) >> inv

LinAlgError: Singular matrix

In [13]:
(np.array([[1,3], [1,2], [0,1]]) >> PP) @ (np.array([[1,0,2], [0,1,2]]) >> PP ) >> PP;

[[1 3]
 [1 2]
 [0 1]]
[[1 0 2]
 [0 1 2]]
[[1 3 8]
 [1 2 6]
 [0 1 2]]


$$
\begin{align}
A = 
\begin{pmatrix}
1 & 4 & 7 \\
2 & 5 & 8 \\
3 & 6 & 9 
\end{pmatrix}
=
\begin{pmatrix}
1 & 4 \\
2 & 5 \\
3 & 6
\end{pmatrix}
\begin{pmatrix}
1 & 4 \\
2 & 5 
\end{pmatrix}^{-1}
\begin{pmatrix}
1 & 4 & 7 \\
2 & 5 & 8 
\end{pmatrix}
\end{align}
$$

p8, Recommended, verify that:

$$
\begin{align}
\begin{pmatrix}
1 & 4 \\
2 & 5 
\end{pmatrix}^{-1}
\begin{pmatrix}
1 & 4 & 7 \\
2 & 5 & 8 
\end{pmatrix}
=
\begin{pmatrix}
1 & 0 & -1 \\
0 & 1 & 2 
\end{pmatrix}
\end{align}
$$


In [14]:
A = matrix([1,4,7],[2,5,8],[3,6,9])
C = A >> cols(_,1,2)
sigma = A >> cols(_,1,2) >> rows(_,1,2) >> PP
r = A >> rows(_,1,2)
R = sigma >> inv >> mmul >> r
with np.printoptions(precision=2, suppress=True): 
    R >> PP
None;

[[1 4] 
 [2 5]]
[[ 1.  0. -1.] 
 [ 0.  1.  2.]]


p8, Highly Recommended: Create a 3 by 3 matrix A wit rant 1. Factor A into A = CR



In [15]:
A = matrix([3, 2, 1], [6, 4, 2], [9,6,3]) >> PP
r = A >> row(_,1) >> PP
sigma = matrix([3]) >> PP
R = (sigma >> inv) >> mmul >> r >> PP
with np.printoptions(precision=2, suppress=True):
    TR(A >> col(_,1) >> mmul >> R) >> PP
None;

[[3 2 1] 
 [6 4 2] 
 [9 6 3]]
[[3 2 1]]
[[3]]
[[1.         0.66666667 0.33333333]]
[[3. 2. 1.] 
 [6. 4. 2.] 
 [9. 6. 3.]]


In [16]:
TR(2) >> join >> TR(" * ") >> join >> TR(matrix([1],[2],[3])) >> join >> TR("  +   ") >> join >> TR('x2 * ') >> join >> TR(matrix([1],[2],[3])) >> PP;

    [[1]            [[1] 
2 *  [2]   +   x2 *  [2] 
     [3]]            [3]]


In [17]:
matrix([2],[2],[1]) >> mmul >> matrix([3,4,6]) >> PP;

[[ 6  8 12] 
 [ 6  8 12] 
 [ 3  4  6]]


In [18]:
A = matrix([1,0],[3,1])
B = matrix([2,4],[0,5])
TR(col(A,1)) >> join >> TR(row(B,1)) >> join >> TR(" + ") >> join >> TR(col(A,2)) >> join >> TR(row(B,2)) >> PP;
TR(col(A,1) >> mmul >>row(B,1)) >> join >> TR("   +  ") >> join >> TR(col(A,2)>> mmul >> row(B,2)) >> PP;
TR((col(A,1) >> mmul >>row(B,1)) >> madd >> (col(A,2)>> mmul >> row(B,2))) >> PP;
TR(A >> mmul >> B) >> PP;

[[1] [[2 4]] + [[0] [[0 5]]
 [3]]           [1]]       
[[ 2  4]    +  [[0 0] 
 [ 6 12]]       [0 5]]
[[ 2  4] 
 [ 6 17]]
[[ 2  4] 
 [ 6 17]]


p12, Spectral Theorem

$$
\begin{align}
%\begin{split}
S^\top &= S
            && \text{where $S$ is N x N} \\
Q^\top &= Q^{-1} \\
S &= Q \Lambda Q^\top \\
S q_i &= \lambda_i q_i
            && \text{where $q_i$ is N x 1} \\
S Q &= Q \Lambda 
            && \text{post multiple by $Q^{-1} = Q^\top} \\
S &= Q \Lambda Q^\top
%\end{split}
\end{align}
$$


show that $S q_i = \lambda_i q_i$

In [19]:
AB = A >> mmul >> B
vals, vecs = np.linalg.eig(AB)
TR(AB >> mmul >> col(vecs, 1)) >> join >> TR('  =  ') >> join >> TR(vals[0] * col(vecs, 1)) >> PP
TR(AB >> mmul >> col(vecs, 2)) >> join >> TR(' =  ') >> join >> TR(vals[1] * col(vecs, 2)) >> PP;

[[-0.50899482]   =  [[-0.50899482] 
 [ 0.1855587 ]]      [ 0.1855587 ]]
[[ -4.35918143]  =  [[ -4.35918143] 
 [-17.93610965]]     [-17.93610965]]
